##Silver Layer Injestion

In [0]:
%sql
USE CATALOG olist_ecommerce_project;

####Importing Libraries

In [0]:
from pyspark.sql.functions import col, sum as spark_sum
from pyspark.sql.functions import regexp_extract, regexp_replace, trim, lower, when

### Sellers Table Data Manipulation and Cleaning

In [0]:
df_sellers_bronze = spark.table("olist_ecommerce_project.bronze.brz_sellers")

# Basic profiling
print("Total rows:", df_sellers_bronze.count())


Total seller_id count

In [0]:
print("Distinct seller_id:", df_sellers_bronze.select("seller_id").distinct().count())

Checking null values in the columns

In [0]:
# Null check
df_sellers_bronze.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_sellers_bronze.columns
]).show()

Checking the casing letter of columns

In [0]:
# Check city/state casing
df_sellers_bronze.select("seller_city").distinct().orderBy("seller_city").show(30, truncate=False)
df_sellers_bronze.select("seller_state").distinct().orderBy("seller_state").show(30, truncate=False)

seller_city column have unresolved rows like having numeric values, truncated values and also some rows have state abbreviation as part of name

In [0]:
# Checking the issues deeply

# How many cities look like zip codes (all digits)?
print("Cities that look like zip codes:")
df_sellers_bronze.filter(col("seller_city").rlike("^[0-9]+$")).select("seller_city").show(10, truncate=False)

print("Count:", df_sellers_bronze.filter(col("seller_city").rlike("^[0-9]+$")).count())


In [0]:
# How many cities have state codes appended (end with 2-letter state like ' sp', ' rj' etc)?
print("\nCities with state codes appended:")
df_sellers_bronze.filter(col("seller_city").rlike(r" [a-z]{2}$")).select("seller_city").show(20, truncate=False)

print("Count:", df_sellers_bronze.filter(col("seller_city").rlike(r" [a-z]{2}$")).count())

#### Fixing the issues and creating the seller silver table

In [0]:
df_sellers_silver = df_sellers_bronze.drop("_source_file")

# Step 1: Null out emails — contains '@'
df_sellers_silver = df_sellers_silver.withColumn(
    "seller_city",
    when(col("seller_city").rlike("@"), None)
    .otherwise(col("seller_city"))
)

# Step 2: Null out purely numeric values (zip codes in wrong column)
df_sellers_silver = df_sellers_silver.withColumn(
    "seller_city",
    when(col("seller_city").rlike("^[0-9]+$"), None)
    .otherwise(col("seller_city"))
)

# Step 3: Remove everything inside parentheses including the parentheses
# 'arraial d'ajuda (porto seguro)' → 'arraial d'ajuda'
df_sellers_silver = df_sellers_silver.withColumn(
    "seller_city",
    trim(regexp_replace(col("seller_city"), r"\s*\(.*?\)", ""))
)

# Step 4: Remove everything after these separators: / \ , -
# This is fully dynamic — doesn't care what comes after the separator
# 'jacarei / sao paulo' → 'jacarei'
# 'novo hamburgo, rio grande do sul, brasil' → 'novo hamburgo'
# 'lages - sc' → 'lages'
# 'rio de janeiro \rio de janeiro' → 'rio de janeiro'
df_sellers_silver = df_sellers_silver.withColumn(
    "seller_city",
    trim(regexp_replace(col("seller_city"), r"\s*[/\\,\-]\s*.*$", ""))
)

# Step 5: Remove trailing 2-letter state codes with space separator
# 'angra dos reis rj' → 'angra dos reis'
# 'brasilia df' → 'brasilia'
df_sellers_silver = df_sellers_silver.withColumn(
    "seller_city",
    trim(regexp_replace(col("seller_city"), r"\s+[a-z]{2}$", ""))
)

# Step 6: Null out anything 2 letters or less remaining after all cleaning
# e.g. 'sp / sp' → 'sp' → null
df_sellers_silver = df_sellers_silver.withColumn(
    "seller_city",
    when(col("seller_city").rlike("^[a-z]{0,2}$"), None)
    .otherwise(col("seller_city"))
)

# Sanity check before writing
df_sellers_silver.select("seller_city").distinct().orderBy("seller_city").show(50, truncate=False)

In [0]:
# Method 1: Count null values (simple)
null_count = df_sellers_silver.filter(col("seller_city").isNull()).count()
print(f"Number of nulls in seller_city: {null_count}")

# Method 2: Show rows with null values
df_sellers_silver.filter(col("seller_city").isNull()).show(20, truncate=False)

In [0]:

# Replace 70000 with your specific zip code
specific_zip = "12903"  # Change this to your zip code

# Get all distinct cities for this zip code
df_specific_zip = df_sellers_silver.filter(col("seller_zip_code_prefix") == specific_zip)

# Show all cities associated with this zip code
df_specific_zip.select("seller_city", "seller_zip_code_prefix") \
    .distinct() \
    .orderBy("seller_city") \
    .show(50, truncate=False)

# Count how many cities this zip code has
num_cities = df_specific_zip.select("seller_city").distinct().count()
print(f"\nZip code {specific_zip} has {num_cities} different cities")

Adding values to the new Table Sellers Slv Table

In [0]:
# Write to Silver
(
    df_sellers_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist_ecommerce_project.silver.slv_sellers")
)